In [ ]:
# Feature Distribution and Outlier Detection

#Analisis ini memvalidasi seluruh fitur yang tersedia dan mendeteksi outlier menggunakan pendekatan statistik standar.
#</VSCode.Cell>
#<VSCode.Cell language="python">
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set(style="whitegrid")

project_root = Path.cwd()
for candidate in [project_root, project_root / "Analysis", project_root.parent]:
    if (candidate / "processed").exists():
        project_root = candidate
        break

processed_dir = project_root / "processed"
feature_names_path = processed_dir / "feature_names.json"
X_path = processed_dir / "X.npy"
metadata_path = processed_dir / "metadata.csv"
y_path = processed_dir / "y.npy"

if not feature_names_path.exists() or not X_path.exists():
    raise FileNotFoundError("File feature_names.json atau X.npy tidak ditemukan di folder processed. Jalankan build_all_processed_dataset.py terlebih dahulu.")

with open(feature_names_path, "r", encoding="utf-8") as f:
    feature_names = json.load(f)

X = np.load(X_path)
y = np.load(y_path)
if X.ndim == 3:
    X = X.reshape(-1, X.shape[2])
elif X.ndim == 1:
    X = X.reshape(-1, 1)

if len(feature_names) != X.shape[1]:
    raise ValueError(
        f"Jumlah fitur ({len(feature_names)}) tidak cocok dengan dimensi X ({X.shape[1]})."
    )

feature_df = pd.DataFrame(X, columns=feature_names)
metadata = pd.read_csv(metadata_path)
if len(metadata) != len(feature_df):
    raise ValueError(f"Jumlah baris metadata ({len(metadata)}) tidak cocok dengan fitur ({len(feature_df)}).")

feature_df = pd.concat([feature_df, metadata[["Subject", "Night", "Epoch"]]], axis=1)
feature_df["label"] = y.astype(int)
print("Loaded feature matrix:", feature_df.shape)
print("Subjects:", sorted(feature_df["Subject"].unique().tolist()))
print(feature_df.describe().T)
#</VSCode.Cell>
#<VSCode.Cell language="python">
# Distribusi fitur dan outlier detection

fig, axes = plt.subplots(6, 4, figsize=(20, 24), constrained_layout=True)
axes = axes.flatten()

plot_columns = [
    col for col in feature_df.select_dtypes(include=[np.number]).columns
    if col not in {"Subject", "Night", "Epoch", "label"}
]

outlier_summary = []
for idx, column in enumerate(plot_columns):
    ax = axes[idx]
    values = pd.to_numeric(feature_df[column], errors="coerce").dropna()
    if values.empty:
        ax.axis("off")
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    if np.isclose(iqr, 0.0):
        lower = q1 - 1.5
        upper = q3 + 1.5
    else:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

    outlier_mask = (values < lower) | (values > upper)
    outliers = values[outlier_mask]

    sns.histplot(values, kde=True, ax=ax, stat="density", color="tab:blue", edgecolor="white", linewidth=0.5)
    ax.axvline(lower, color="red", linestyle="--", linewidth=1)
    ax.axvline(upper, color="red", linestyle="--", linewidth=1)
    ax.set_title(f"{column}\nOutliers: {outliers.size}")
    ax.set_xlabel("")
    ax.set_ylabel("Density")

    outlier_summary.append({
        "feature": column,
        "n_outliers": int(outliers.size),
        "lower_bound": float(lower),
        "upper_bound": float(upper),
        "q1": float(q1),
        "q3": float(q3),
    })

for ax in axes[len(plot_columns):]:
    ax.axis("off")

plt.suptitle("Feature Distributions with IQR Outlier Boundaries", fontsize=20, y=0.95)
plt.show()

outlier_df = pd.DataFrame(outlier_summary).sort_values("n_outliers", ascending=False)
print(outlier_df)

outlier_features = outlier_df[outlier_df["n_outliers"] > 0]
print("\nFitur dengan outlier terdeteksi:")
print(outlier_features[["feature", "n_outliers"]].to_string(index=False))


Loaded feature matrix: (65027, 26)
Subjects: ['Bidslab00', 'Bidslab01', 'Bidslab02', 'Bidslab06', 'Bidslab07', 'Bidslab08', 'Bidslab09', 'Bidslab10', 'Bidslab11', 'Bidslab13', 'Bidslab14', 'Bidslab15', 'Bidslab16', 'Bidslab17', 'Bidslab18', 'Bidslab19']
                     count         mean         std           min  \
mean_ibi           65027.0   927.850159  103.428085  5.548160e+02   
mean_hr            65027.0    65.534164    7.881238  4.953407e+01   
sdnn               65027.0    37.834717   25.965969  0.000000e+00   
rmssd              65027.0    25.249952   20.330830  0.000000e+00   
sdsd               65027.0    25.205048   20.316334  0.000000e+00   
nn50               65027.0     4.154228    9.524305  0.000000e+00   
pnn50              65027.0     0.063620    0.094474  0.000000e+00   
lf                 65027.0     4.060217    1.348747  0.000000e+00   
hf                 65027.0     0.294008    1.440418  0.000000e+00   
lf_hf              65027.0     6.533678    1.007626  0.0